# gemma4-swe-rl — Colab free (T4, 16GB)

Прогон пайплайна на суррогате семейства Gemma-4 (bf16, без квантизации): mock-логика → реальный vLLM + gemma4 tool-calling parser → полный эпизод через sim/ → мини-прогон GiGPO.
Модель малая и не боевая: цель — механика и формат, не качество фиксов.

In [ ]:
import os
os.environ["SWE_ENV"] = "colab_free"

!git clone https://github.com/aletonfdg/gemma4-swe-rl.git
%cd gemma4-swe-rl
!pip install -q -r requirements.txt vllm

In [ ]:
!python inspect_harness.py
# если есть доступ к датасету -- хотя бы несколько снапшотов для реального verify_task
!python inspect_sample_submission.py data/

In [ ]:
import os
os.environ["SWE_ENV"] = "local"   # mock backend, не трогает GPU вообще
# ТРЕБУЕТ: scripts/dev_smoke_test.py -- в репозитории пока нет каталога scripts/
!python scripts/dev_smoke_test.py

In [ ]:
import os
os.environ["SWE_ENV"] = "colab_free"

!python inspect_harness.py  # повторно -- под новым конфигом

# Поднять сервер и сделать один ручной запрос с tools=
from llm.vllm_server import VLLMServerProcess
from llm.llm_client import LLMClient
from sim.tool_schemas import ROOT_TOOL_SCHEMAS

# ВНИМАНИЕ: VLLMServerProcess.start() жёстко ставит --quantization compressed-tensors
# (llm/vllm_server.py:24) и берёт model_id из config.yaml -- на bf16-суррогате упадёт.
server = VLLMServerProcess()
server.start(lora_paths=None)
client = LLMClient(model="gemma-4-2b-it")

resp = client.chat(
    messages=[{"role": "user", "content": "List files in /workspace using run_command."}],
    tools=ROOT_TOOL_SCHEMAS,
    temperature=0.2,
)
print(resp)
server.stop()

In [ ]:
import yaml
from sim.swe_env import SWEEnv
from rl.rollout import run_episode
from llm.llm_client import LLMClient
from llm.vllm_server import VLLMServerProcess

# NOTE: config_loader.CFG в репозитории нет -- конфиг читается из config.yaml
CFG = yaml.safe_load(open("config.yaml"))

# NOTE: SWEEnv всегда поднимает DockerSandbox -- sandbox_backend: subprocess не реализован
env = SWEEnv("data/tasks.jsonl", "data/snapshots", "data/graphs", "data/embeddings")
client = LLMClient(model="gemma-4-2b-it")
server = VLLMServerProcess(); server.start(lora_paths=None)

steps, messages, passed = run_episode(env, 0, client, "data/snapshots", CFG["docker"])
print(len(steps), passed)
server.stop()

In [ ]:
import os
os.environ["SWE_ENV"] = "colab_free"
!python train_gigpo.py